## Imports

In [ ]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../../'))

import neuro_fuzzy_toolbox as nft

In [7]:
import numpy as np

In [8]:
from sklearn.preprocessing import MinMaxScaler

# Surface (1k)

## Data

In [9]:
def z(x, y):
  return ((3) * ((1-x)**2) * (np.exp(-(x**2)-((y+1)**2))) - (10) * ((x/5)-(x**3)-(y**5)) * (np.exp(-(x**2)-(y**2))) - (1/3)*np.exp(-(x+1)**2-(y**2)))

#Training
x0 = np.random.uniform(-3,3,1000)
x1 = np.random.uniform(-3,3,1000)

e = np.random.normal(0,0.5,1000) #noise
Y = z(x0,x1) + e

#Testing
x0_test = np.random.uniform(-3,3,1000)
x1_test = np.random.uniform(-3,3,1000)

Y_test = z(x0_test,x1_test)

In [10]:
#Training
scaler = MinMaxScaler(feature_range=(0, 1))
vstack_train = np.vstack((x0,x1)).T
scaled_train = scaler.fit_transform(vstack_train)

#Testing
vstack_test = np.vstack((x0_test,x1_test)).T
scaled_test = scaler.transform(vstack_test)

In [11]:
dtype = torch.float64

train_loader = data.DataLoader(
    data.TensorDataset(
        torch.tensor(scaled_train, dtype=dtype), 
        torch.tensor(Y, dtype=dtype)), 
    batch_size = 8,
    shuffle = True)

x_train = train_loader.dataset.tensors[0]
y_train = train_loader.dataset.tensors[1]

x_test = torch.tensor(scaled_test, dtype=dtype)
y_test = torch.tensor(Y_test, dtype=dtype)

In [12]:
x_train.dtype

torch.float64

## Model & Training

### ANFIS

In [13]:
model = nft.rule_reduced_ANFIS(
    input_size = 2,
    num_mfs = 2,
    outputs = 1,
    dtype=torch.float64
)

In [14]:
model.init_premises(x_train)

### Hybrid Learning Algorithm

In [15]:
loss_fn = nn.functional.mse_loss
#loss_fn = nn.functional.binary_cross_entropy
#loss_fn = nn.functional.cross_entropy

optimizer = torch.optim.AdamW
params = {'lr': 0.0001, 'weight_decay': 0.001}

early_stopping = nft.EarlyStopping(patience=30, delta=0.001)

In [16]:
trainer = nft.Hybrid_learning_algorithm(
    epochs=500,
    loss_function=loss_fn,
    optimizer=optimizer,
    optimizer_params=params,
    early_stopping=early_stopping
)

### SONFIS

In [17]:
Ngrow = 60
dGrow = 0.7
Nsplit = 50
eSplit = 0.8
Nvanish = 6
lVanish = 3

max_iterations = 40

anfis_trainer = trainer

validation = 0.3
sonfis_early_stopping = nft.EarlyStopping(patience=8)
last_training_iteration = True

In [18]:
sonfis = nft.SONFIS(
    Ngrow=Ngrow,
    dGrow=dGrow,
    Nsplit=Nsplit,
    eSplit=eSplit,
    Nvanish=Nvanish,
    lVanish=lVanish,
    max_iterations=max_iterations,
    ANFIStrainer=anfis_trainer,
    validation=validation,
    early_stopping=sonfis_early_stopping,
    last_training_iteration=last_training_iteration
)

In [19]:
%time sonfis(model, train_loader, verbose=True)

Iteration:  0/40 - loss: 5.288930 - validation loss: 5.436092
 -> ANFIS rules: 2

Iteration:  1/40 - loss: 1.066591 - validation loss: 1.182006
 -> ANFIS rules: 4

Iteration:  2/40 - loss: 1.184278 - validation loss: 1.233856
 -> ANFIS rules: 5

Iteration:  3/40 - loss: 11.808867 - validation loss: 12.459494
 -> ANFIS rules: 6

Iteration:  4/40 - loss: 6.301907 - validation loss: 6.464344
 -> ANFIS rules: 8

Iteration:  5/40 - loss: 9.291879 - validation loss: 9.710010
 -> ANFIS rules: 12

Iteration:  6/40 - loss: 0.558863 - validation loss: 0.823402
 -> ANFIS rules: 17

Iteration:  7/40 - loss: 0.554217 - validation loss: 0.781779
 -> ANFIS rules: 19

Iteration:  8/40 - loss: 0.562935 - validation loss: 0.800215
 -> ANFIS rules: 17

No more updates
Iteration:  9/40 - loss: 0.562935 - validation loss: 0.800215
 -> ANFIS rules: 17

Last training iteration
Iteration: 10/40 - loss: 0.203602 - validation loss: 0.316393

Training finished
 -> ANFIS rules: 17

CPU times: user 2min 56s, sys: 

In [20]:
test_measures = nft.get_measures(model, x_test, y_test)

for measure in test_measures:
    print(measure + ':', test_measures[measure])

MSE: 0.09113940116999032
RMSE: 0.3018930293497853
MAE: 0.21908857003636314
R2: 0.9746556494776948
MAPE: 55.13827362291791


In [21]:
train_measures = nft.get_measures(model, x_train, y_train)

for measure in train_measures:
    print(measure + ':', train_measures[measure])

MSE: 0.23743952156860043
RMSE: 0.4872776637283926
MAE: 0.38505914082359655
R2: 0.9350625468899463
MAPE: 3.1897002048084357
